# 🍏 Observability & Tracing Demo with `azure-ai-projects` and `azure-ai-inference` 🍎

Welcome to this **Health & Fitness**-themed notebook, where we'll explore how to set up **observability** and **tracing** for:

1. **Basic LLM calls** using an `AIProjectClient`.
2. **Multi-step** interactions using an **Agent** (such as a Health Resource Agent).
3. **Tracing** your local usage in **console** (stdout) or via an **OTLP endpoint** (like **Prompty** or **Aspire**).
4. Sending those **traces** to **Azure Monitor** (Application Insights) so you can view them in **Azure AI Foundry**.

> **Disclaimer**: This is a fun demonstration of AI and observability! Any references to workouts, diets, or health routines in the code or prompts are purely for **educational** purposes. Always consult a professional for health advice.

## Contents
1. **Initialization**: Setting up environment, creating clients.
2. **Basic LLM Call**: Quick demonstration of retrieving model completions.
3. **Connections**: Listing project connections.
4. **Observability & Tracing**
   - **Console / Local** tracing
   - **Prompty / Aspire**: piping traces to a local OTLP endpoint
   - **Azure Monitor** tracing: hooking up to Application Insights
   - **Verifying** your traces in Azure AI Foundry
5. **Agent-based Example**:
   - Creating a simple "Health Resource Agent" referencing sample docs.
   - Multi-turn conversation with tracing.
   - Cleanup.

<img src="./seq-diagrams/1-observability.png" width="50%"/>

## 1. Initialization & Setup
**Prerequisites**:
- A `.env` file containing `PROJECT_ENDPOINT` (and optionally `MODEL_DEPLOYMENT_NAME`).
- Roles/permissions in Microsoft Foundry that let you do inference & agent creation.
- A local environment with `azure-ai-projects` (2.x), `openai`, and `opentelemetry` packages installed.

**What we do**:
- Load environment variables.
- Initialize the endpoint-based `AIProjectClient` and its OpenAI client.
- Check that we can talk to a model (like `gpt-5.4`).

In [ ]:
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

# Initialize the endpoint-based AIProjectClient + its OpenAI client
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully created AIProjectClient + OpenAI client!")
except Exception as e:
    print(f"❌ Error creating AIProjectClient: {e}")

## 2. Basic LLM Call
We'll do a **quick** chat completion request to confirm everything is working. We'll ask a simple question: "How many feet are in a mile?"

In [ ]:
try:
    # Default to "gpt-5.4" if no env var is set
    model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4")

    user_question = "How many feet are in a mile?"
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": user_question}]
    )
    print("\n💡Response:")
    print(response.choices[0].message.content)
    print("\nFinish reason:", response.choices[0].finish_reason)

except Exception as e:
    print("❌ Could not complete the chat request:", e)

## 3. List & Inspect Connections
Check out the **connections** your project has: these might be Azure OpenAI or other resource attachments. We'll just list them here for demonstration.

In [ ]:
# List all connections in the project
all_conns = list(project_client.connections.list())
print(f"🔎 Found {len(all_conns)} total connections.")
for idx, c in enumerate(all_conns):
    print(f"{idx+1}) Name: {c.name}, Type: {c.type}, Target: {c.target}")

# Filter for Azure OpenAI connections (connection_type accepts the string category)
aoai_conns = list(project_client.connections.list(connection_type="AzureOpenAI"))
print(f"\n🌀 Found {len(aoai_conns)} Azure OpenAI connections:")
for c in aoai_conns:
    print(f"   -> {c.name}")

# 4. Observability & Tracing

We want to **collect telemetry** from our LLM calls, for example:
- Timestamps of requests.
- Latency.
- Potential errors.
- Optionally, the actual prompts & responses (if you enable content recording).

We'll show how to set up:
1. **Console** or local OTLP endpoint instrumentation.
2. **Azure Monitor** instrumentation with Application Insights.
3. **Viewing** your traces in Azure AI Foundry's portal.

## 4.1 Local Console Debugging
We'll install instrumentation packages and enable them. Then we'll do a quick chat call to see if logs appear in **stdout**.

**Note**: If you want to see more advanced local dashboards, you can:
- Use [Prompty](https://github.com/microsoft/prompty).
- Use [Aspire Dashboard](https://learn.microsoft.com/dotnet/aspire/fundamentals/dashboard/standalone?tabs=bash) to visualize your OTLP traces.

In [ ]:
# You only need to install these once.
!pip install opentelemetry-instrumentation-openai-v2 opentelemetry-exporter-otlp-proto-grpc

### 4.1.1 Enable OpenTelemetry for the OpenAI client
We instrument the **OpenAI** client (the one returned by `project_client.get_openai_client()`) so its chat/response calls emit OpenTelemetry spans. We:
1. Optionally capture **prompt & completion content** in traces.
2. Call `OpenAIInstrumentor().instrument()` to patch and enable the instrumentation.


In [ ]:
import os
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

# (Optional) capture prompt & completion contents in traces
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"  # or 'false'

# Instrument the OpenAI client library so its calls emit OpenTelemetry spans.
# (We use the OpenAI client from project_client.get_openai_client(), so we
# instrument OpenAI rather than the retired azure-ai-inference client.)
OpenAIInstrumentor().instrument()
print("✅ OpenAI instrumentation enabled.")

### 4.1.2 Point Traces to Console or Local OTLP
The simplest is to pipe them to **stdout**. If you want to send them to **Prompty** or **Aspire**, specify the local OTLP endpoint URL (usually `"http://localhost:4317"` or similar).

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Pipe spans to stdout (console). To send to Prompty/Aspire instead, replace the
# ConsoleSpanExporter with an OTLPSpanExporter pointed at http://localhost:4317.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

try:
    user_prompt = "What's a simple 5-minute warmup routine?"
    local_resp = openai_client.chat.completions.create(
        model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
        messages=[{"role": "user", "content": user_prompt}]
    )
    print("\n🤖 Response:", local_resp.choices[0].message.content)
except Exception as exc:
    print(f"❌ Error in local-tracing example: {exc}")

## 4.2 Azure Monitor Tracing (Application Insights)
Now we'll set up tracing to **Application Insights**, which will forward your logs to the **Azure AI Foundry** **Tracing** page.

**Steps**:
1. In AI Foundry, go to your project’s **Tracing** tab, attach (or create) an **Application Insights** resource.
2. In code, call `project_client.telemetry.get_connection_string()` to retrieve the instrumentation key.
3. Use `azure.monitor.opentelemetry.configure_azure_monitor(...)` with that connection.
4. Make an inference call -> logs appear in the Foundry portal (and in Azure Monitor itself).


In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor

# Set APPLICATIONINSIGHTS_CONNECTION_STRING in your .env — copy it from the
# Application Insights resource attached to your Foundry project's Tracing tab.
app_insights_conn_str = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")
if app_insights_conn_str:
    print("🔧 Found App Insights connection string, configuring Azure Monitor...")
    configure_azure_monitor(connection_string=app_insights_conn_str)

    # Let's do a test call that logs to AI Foundry's Tracing page
    try:
        prompt_msg = "Any easy at-home cardio exercise recommendations?"
        response = openai_client.chat.completions.create(
            model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
            messages=[{"role": "user", "content": prompt_msg}]
        )
        print("\n🤖 Response (logged to App Insights):")
        print(response.choices[0].message.content)
    except Exception as e:
        print("❌ Chat completion with Azure Monitor example failed:", e)
else:
    print("No APPLICATIONINSIGHTS_CONNECTION_STRING set. Attach Application Insights on the "
          "Foundry Tracing tab and add its connection string to your .env.")

### 4.3 Viewing Traces in Azure AI Foundry
After running the above code:
1. Go to your AI Foundry project.
2. Click **Tracing** on the sidebar.
3. You should see the logs from your calls.
4. Filter, expand, or explore them as needed.

Also, if you want more advanced dashboards, you can open your **Application Insights** resource from the Foundry. In the App Insights portal, you get additional features like **end-to-end transaction** details, query logs, etc.


# 5. Agent-based Example
We'll now create a **Health Resource Agent** that references sample docs about recipes or guidelines, then demonstrate:
1. Creating an Agent with instructions.
2. Creating a conversation thread.
3. Running multi-step queries with **observability** enabled.
4. Optionally cleaning up resources at the end.

> The agent approach is helpful when you want more sophisticated conversation flows or **tool usage** (like file search).

## 5.1 Create Sample Files & Vector Store
We'll create dummy `.md` files about recipes/guidelines, then push them into a **vector store** so our agent can do semantic search.

(*This portion is a quick summary—see [the other file-search tutorial] if you need more details.)

In [ ]:
def create_sample_files():
    """Create some local .md files with sample text."""
    recipes_md = (
        """# Healthy Recipes Database\n\n"
        "## Gluten-Free Recipes\n"
        "1. Quinoa Bowl\n"
        "   - Ingredients: quinoa, vegetables, olive oil\n"
        "   - Instructions: Cook quinoa, add vegetables\n\n"
        "2. Rice Pasta\n"
        "   - Ingredients: rice pasta, mixed vegetables\n"
        "   - Instructions: Boil pasta, sauté vegetables\n\n"
        "## Diabetic-Friendly Recipes\n"
        "1. Low-Carb Stir Fry\n"
        "   - Ingredients: chicken, vegetables, tamari sauce\n"
        "   - Instructions: Cook chicken, add vegetables\n\n"
        "## Heart-Healthy Recipes\n"
        "1. Baked Salmon\n"
        "   - Ingredients: salmon, lemon, herbs\n"
        "   - Instructions: Season salmon, bake\n\n"
        "2. Mediterranean Bowl\n"
        "   - Ingredients: chickpeas, vegetables, tahini\n"
        "   - Instructions: Combine ingredients\n"""
    )

    guidelines_md = (
        """# Dietary Guidelines\n\n"
        "## General Guidelines\n"
        "- Eat a variety of foods\n"
        "- Control portion sizes\n"
        "- Stay hydrated\n\n"
        "## Special Diets\n"
        "1. Gluten-Free Diet\n"
        "   - Avoid wheat, barley, rye\n"
        "   - Focus on naturally gluten-free foods\n\n"
        "2. Diabetic Diet\n"
        "   - Monitor carbohydrate intake\n"
        "   - Choose low glycemic foods\n\n"
        "3. Heart-Healthy Diet\n"
        "   - Limit saturated fats\n"
        "   - Choose lean proteins\n"""
    )

    with open("recipes.md", "w", encoding="utf-8") as f:
        f.write(recipes_md)
    with open("guidelines.md", "w", encoding="utf-8") as f:
        f.write(guidelines_md)

    print("📄 Created sample resource files: recipes.md, guidelines.md")
    return ["recipes.md", "guidelines.md"]

sample_files = create_sample_files()

def create_vector_store(files, store_name="my_health_resources"):
    try:
        # Create the vector store, then upload+attach each file (create_and_poll waits for ingestion)
        vs = openai_client.vector_stores.create(name=store_name)
        print(f"🎉 Created vector store '{store_name}', ID: {vs.id}")
        uploaded_ids = []
        for fp in files:
            with open(fp, "rb") as fh:
                uploaded = openai_client.files.create(purpose="assistants", file=fh)
            openai_client.vector_stores.files.create_and_poll(vector_store_id=vs.id, file_id=uploaded.id)
            uploaded_ids.append(uploaded.id)
            print(f"✅ Uploaded & indexed: {fp} -> File ID: {uploaded.id}")
        return vs, uploaded_ids
    except Exception as e:
        print(f"❌ Error creating vector store: {e}")
        return None, []

vector_store, file_ids = None, []
if sample_files:
    vector_store, file_ids = create_vector_store(sample_files, store_name="health_resources_example")

## 5.2 Create a Health Resource Agent
We'll create a **FileSearchTool** referencing the vector store, then create an agent with instructions that it should:
1. Provide disclaimers.
2. Offer general nutrition or recipe tips.
3. Cite sources if possible.
4. Encourage professional consultation for deeper medical advice.


In [ ]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

def create_health_agent(vs_id):
    try:
        instructions = """
            You are a health resource advisor with access to dietary and recipe files.
            You:
            1. Always present disclaimers (you're not a medical professional)
            2. Provide references to files when possible
            3. Focus on general nutrition or recipe tips.
            4. Encourage professional consultation for more detailed advice.
        """

        agent = project_client.agents.create_version(
            agent_name="health-search-agent",
            definition=PromptAgentDefinition(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                instructions=instructions,
                tools=[FileSearchTool(vector_store_ids=[vs_id])],
            ),
            description="Health resource search agent over uploaded documents.",
        )
        print(f"🎉 Created agent '{agent.name}', version: {agent.version}")
        return agent
    except Exception as e:
        print(f"❌ Error creating health agent: {e}")
        return None

health_agent = None
if vector_store:
    health_agent = create_health_agent(vector_store.id)

## 5.3 Using the Agent
Let's create a new conversation **thread** and ask the agent some questions. We'll rely on the **observability** settings we already configured so each step is traced.


In [ ]:
def ask_question(agent, conversation_id, user_question):
    try:
        print(f"User asked: '{user_question}'")
        # Run the agent via the Responses API (traced by the instrumentation above)
        response = openai_client.responses.create(
            conversation=conversation_id,
            input=user_question,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"Response status: {response.status}")
        return response
    except Exception as e:
        print(f"❌ Error asking question: {e}")
        return None

conversation = None
responses = []
if health_agent:
    conversation = openai_client.conversations.create()
    print(f"📝 Created new conversation, ID: {conversation.id}")
    # Let's ask a few sample questions
    queries = [
        "Could you suggest a gluten-free lunch recipe?",
        "Show me some heart-healthy meal ideas.",
        "What guidelines do you have for someone with diabetes?"
    ]
    for q in queries:
        r = ask_question(health_agent, conversation.id, q)
        if r:
            responses.append((q, r))

### 5.3.1 Viewing the conversation
We can retrieve the conversation messages to see how the agent responded, check if it cited file passages, etc.

In [ ]:
def display_results(pairs):
    try:
        print("\n🗣️ Conversation:")
        for user_question, response in pairs:
            print(f"[USER]: {user_question}")
            print(f"[ASSISTANT]: {response.output_text}\n")

            # Surface any file citations from the response annotations
            for item in (response.output or []):
                if getattr(item, "type", "") != "message":
                    continue
                for content in (getattr(item, "content", None) or []):
                    for ann in (getattr(content, "annotations", None) or []):
                        if getattr(ann, "type", "") == "file_citation":
                            print(f"   📎 Citation: {getattr(ann, 'filename', '') or getattr(ann, 'file_id', '')}")
    except Exception as e:
        print(f"❌ Could not display results: {e}")

# If we asked questions above, display the answers + citations
if responses:
    display_results(responses)

# 6. Cleanup
If desired, we can remove the vector store, files, and agent to keep things tidy. (In a real solution, you might keep them around.)

In [ ]:
def cleanup_resources():
    try:
        if 'health_agent' in globals() and health_agent:
            project_client.agents.delete_version(
                agent_name=health_agent.name,
                agent_version=health_agent.version,
            )
            print("🗑️ Deleted health agent.")

        if 'vector_store' in globals() and vector_store:
            openai_client.vector_stores.delete(vector_store.id)
            print("🗑️ Deleted vector store.")

        if 'file_ids' in globals() and file_ids:
            for fid in file_ids:
                openai_client.files.delete(fid)
            print("🗑️ Deleted uploaded files.")

        if 'sample_files' in globals() and sample_files:
            for sf in sample_files:
                if os.path.exists(sf):
                    os.remove(sf)
            print("🗑️ Deleted local sample files.")
    except Exception as e:
        print(f"❌ Error cleaning up: {e}")


cleanup_resources()

# 🎉 Wrap-Up
We've demonstrated:
1. **Basic LLM calls** with `AIProjectClient`.
2. **Listing connections** in your Azure AI Foundry project.
3. **Observability & tracing** in both local (console, OTLP endpoint) and cloud (App Insights) contexts.
4. A quick **Agent** scenario that uses a vector store for searching sample docs.

## Next Steps
- Check the **Tracing** tab in your Azure AI Foundry portal to see the logs.
- Explore advanced queries in Application Insights.
- Use [Prompty](https://github.com/microsoft/prompty) or [Aspire](https://learn.microsoft.com/dotnet/aspire/) for local telemetry dashboards.
- Incorporate this approach into your **production** GenAI pipelines!

> 🏋️ **Health Reminder**: The LLM's suggestions are for demonstration only. For real health decisions, consult a professional.

Happy Observing & Tracing! 🎉